# Étape 2 - Partie 2 : Moteur de Recherche (Stratégie Question-Réponse)
**Objectif :** Pour chaque question du jeu de test (`test_unique`), trouver les $k=10$ réponses les plus pertinentes dans la base de connaissances (`train_unique`) en utilisant la similarité cosinus.
**Méthodes :** TF-IDF et Word2Vec.

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Interface graphique pour la sélection de fichiers
import tkinter as tk
from tkinter import filedialog

# Bibliothèques pour le NLP et les mathématiques
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

# ==========================================
# SÉLECTION DES FICHIERS VIA INTERFACE
# ==========================================
root = tk.Tk()
root.attributes('-topmost', True)
root.withdraw()

# 1. Sélection du fichier d'entraînement
print("-> Sélectionner le fichier d'entraînement (train_unique_e2p2.csv)...")
chemin_train = filedialog.askopenfilename(
    title="Sélectionne ton fichier d'entraînement",
    filetypes=[("Fichiers CSV", "*.csv"), ("Tous les fichiers", "*.*")]
)

# 2. Sélection du fichier de test
print("-> Sélectionner le fichier de test (test_unique_e2p2.csv)...")
chemin_test = filedialog.askopenfilename(
    title="Sélectionne ton fichier de test",
    filetypes=[("Fichiers CSV", "*.csv"), ("Tous les fichiers", "*.*")]
)

# ==========================================
# CHARGEMENT DES DONNÉES
# ==========================================
df_train = pd.read_csv(chemin_train)
df_test = pd.read_csv(chemin_test)

print(f"Données chargées ! Train: {len(df_train)} lignes | Test: {len(df_test)} lignes.")

In [ ]:
# ==========================================
# FONCTION DE NETTOYAGE COMMUNE AU GROUPE
# ==========================================

def nettoyage_texte(texte):
    """
    Fonction provisoire, à remplacer par le code d'Idir
    """
    texte_propre = str(texte).lower()
    return texte_propre

# Application du nettoyage sur les questions (test) et les réponses (train)
df_train['Response_clean'] = df_train['Response'].apply(nettoyage_texte)
df_test['Context_clean'] = df_test['Context'].apply(nettoyage_texte)

print("Nettoyage appliqué sur les deux datasets.")

## Méthode 1 : Vectorisation par TF-IDF
Le TF-IDF donne un poids aux mots. On vectorise les réponses de l'entraînement pour créer notre "base de recherche", puis on vectorise les questions du test dans le même espace mathématique pour calculer la distance.

In [ ]:
k = 10 

# 1. Initialisation et création du vocabulaire
tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(df_train['Response_clean'])

# 2. Transformation en vecteurs
vecteurs_train_reponses = tfidf.transform(df_train['Response_clean'])
vecteurs_test_questions = tfidf.transform(df_test['Context_clean'])

print("--- RÉSULTATS DU MOTEUR DE RECHERCHE : TF-IDF ---\n")

for i in range(len(df_test)):
    question_brute = df_test.iloc[i]['Context']
    vecteur_q = vecteurs_test_questions[i]
    
    similarites = cosine_similarity(vecteur_q, vecteurs_train_reponses).flatten()
    indices_top_k = similarites.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} :")
    print(f"   \"{question_brute[:150]}...\"")
    
    # --- Affichage de la meilleure réponse trouvée ---
    meilleur_indice = indices_top_k[0]
    score_max = similarites[meilleur_indice]
    vraie_reponse = df_train.iloc[meilleur_indice]['Response']
    
    print(f"\n   MEILLEURE RÉPONSE TROUVÉE (Score: {score_max:.4f}) :")
    print(f"   \"{vraie_reponse[:300]}...\"") # On affiche les 300 premiers caractères
    
    print(f"\n   Indices du Top {k} : {indices_top_k.tolist()}")
    print("-" * 80)

## Méthode 2 : Vectorisation Sémantique par Word2Vec
Ici, l'IA essaie de comprendre le "sens" des phrases. On transforme chaque mot en coordonnées spatiales, puis on fait la moyenne de ces coordonnées pour obtenir le vecteur global de la phrase.

In [ ]:

phrases_train_w2v = [texte.split() for texte in df_train['Response_clean']]
phrases_test_w2v = [texte.split() for texte in df_test['Context_clean']]
w2v_model = Word2Vec(sentences=phrases_train_w2v, vector_size=100, window=5, min_count=1, workers=4)

def vecteur_phrase(liste_mots, modele):
    vecteurs_mots = [modele.wv[mot] for mot in liste_mots if mot in modele.wv]
    if len(vecteurs_mots) == 0: return np.zeros(modele.vector_size)
    return np.mean(vecteurs_mots, axis=0)

vecteurs_train_w2v = np.array([vecteur_phrase(mots, w2v_model) for mots in phrases_train_w2v])
vecteurs_test_w2v = np.array([vecteur_phrase(mots, w2v_model) for mots in phrases_test_w2v])

print("--- RÉSULTATS DU MOTEUR DE RECHERCHE : WORD2VEC ---\n")

for i in range(len(df_test)):
    question_brute = df_test.iloc[i]['Context']
    vecteur_q_w2v = vecteurs_test_w2v[i].reshape(1, -1)
    
    similarites_w2v = cosine_similarity(vecteur_q_w2v, vecteurs_train_w2v).flatten()
    indices_top_k_w2v = similarites_w2v.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} :")
    print(f"   \"{question_brute[:150]}...\"")
    
    # --- Affichage de la meilleure réponse trouvée ---
    meilleur_indice_w2v = indices_top_k_w2v[0]
    score_max_w2v = similarites_w2v[meilleur_indice_w2v]
    vraie_reponse_w2v = df_train.iloc[meilleur_indice_w2v]['Response']
    
    print(f"\n   MEILLEURE RÉPONSE TROUVÉE (Score: {score_max_w2v:.4f}) :")
    print(f"   \"{vraie_reponse_w2v[:300]}...\"")
    
    print(f"\n   Indices du Top {k} : {indices_top_k_w2v.tolist()}")
    print("-" * 80)

## Méthode 3 : Vectorisation Contextuelle par BERT (Sentence-BERT)
BERT est un modèle d'Intelligence Artificielle de type "Transformer". Contrairement à Word2Vec qui regarde les mots un par un, BERT lit la phrase entière dans les deux sens (bidirectionnel) pour comprendre le contexte exact de chaque mot avant de générer le vecteur mathématique.

In [ ]:
# /!\ installer : !pip install sentence-transformers

# Importation du modèle Sentence-BERT
from sentence_transformers import SentenceTransformer

k = 10

# 1. Chargement du modèle pré-entraîné
# 'all-MiniLM-L6-v2' est un modèle de la famille BERT léger, très rapide et excellent pour l'anglais
print("Chargement du modèle BERT en mémoire...")
modele_bert = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Transformation des textes en vecteurs denses
# L'encodage peut prendre quelques dizaines de secondes selon la puissance de ton ordinateur
print("Encodage des 2700 réponses du Train en cours (cela peut prendre un instant)...")
vecteurs_train_bert = modele_bert.encode(df_train['Response_clean'].tolist())

print("Encodage des questions du Test en cours...")
vecteurs_test_bert = modele_bert.encode(df_test['Context_clean'].tolist())

print("\n--- RÉSULTATS DU MOTEUR DE RECHERCHE : BERT ---\n")

# 3. Calcul de la similarité pour chaque question du test
for i in range(len(df_test)):
    question_brute = df_test.iloc[i]['Context']
    # On redimensionne le vecteur pour le calcul (1 ligne, X colonnes)
    vecteur_q_bert = vecteurs_test_bert[i].reshape(1, -1)
    
    similarites_bert = cosine_similarity(vecteur_q_bert, vecteurs_train_bert).flatten()
    indices_top_k_bert = similarites_bert.argsort()[-k:][::-1]
    
    print(f"QUESTION TEST {i+1} :")
    print(f"   \"{question_brute[:150]}...\"")
    
    # Affichage de la meilleure réponse trouvée
    meilleur_indice_bert = indices_top_k_bert[0]
    score_max_bert = similarites_bert[meilleur_indice_bert]
    vraie_reponse_bert = df_train.iloc[meilleur_indice_bert]['Response']
    
    print(f"\n   MEILLEURE RÉPONSE TROUVÉE (Score: {score_max_bert:.4f}) :")
    print(f"   \"{vraie_reponse_bert[:300]}...\"")
    
    print(f"\n   Indices du Top {k} : {indices_top_k_bert.tolist()}")
    print("-" * 80)